In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.embeddings import Embeddings
from pathlib import Path
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import TextLoader
import glob
from openai import OpenAI
import os
from embed import NVIDIAEmbeddings


In [71]:
NVIDIA_API_KEY=os.environ.get("NVIDIA_API_KEY")
pinecone_api_key=os.environ.get("PINECONE_API_KEY")

In [66]:
index_name="sift"

In [29]:
output_path = "../data/parsed"
pages_path = glob.glob(f"{output_path}/*.md")

In [30]:
pages_path

['../data/parsed/page_001.md',
 '../data/parsed/page_002.md',
 '../data/parsed/page_003.md',
 '../data/parsed/page_004.md',
 '../data/parsed/page_005.md',
 '../data/parsed/page_006.md',
 '../data/parsed/page_007.md',
 '../data/parsed/page_008.md',
 '../data/parsed/page_009.md',
 '../data/parsed/page_010.md',
 '../data/parsed/page_011.md']

In [36]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = []

for page in pages_path:
    text_loader = TextLoader(
        page,
        encoding="utf-8"
    )

    docs.extend(text_loader.load())

chunks = splitter.split_documents(docs)

In [46]:
client = OpenAI(
    api_key=NVIDIA_API_KEY, 
    base_url="https://integrate.api.nvidia.com/v1"
)


response = client.embeddings.create(
    input=[chunk.page_content for chunk in chunks],
    model="nvidia/llama-nemotron-embed-1b-v2",
    encoding_format="float",
    extra_body={
        "input_type": "passage",
        "truncate": "NONE"
    }
)

embedding = response.data[0].embedding

In [49]:
print(len(embedding))
print(embedding[:10])

2048
[0.0101318359375, -0.022735595703125, 0.01006317138671875, 0.029632568359375, -0.0149993896484375, -0.01473236083984375, 0.01297760009765625, 0.0019073486328125, -0.0249786376953125, 0.00785064697265625]


In [54]:
embeddings = NVIDIAEmbeddings(
    client=client,
    model="nvidia/llama-nemotron-embed-1b-v2"
)

In [73]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone("pcsk_5rFpZL_GJS5ZE9rrG7MHxuroDfdtkyM9HamR9yQ686WwmcMRMyjnnkFCmuEEafoWuKGWum")

index = pc.Index(index_name)

In [74]:
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [75]:
vector_store.add_documents(chunks)

['b7758e4f-f5e6-4516-8775-e49e03564e2e',
 '293696d2-4bb9-46d2-97db-544360cb845b',
 'f8509103-4050-4b90-91d9-9726a3b17af2',
 '479ab2e0-b1fa-4aff-af63-54632b55b2d1',
 '36f4a46c-64d6-4b88-8bd7-c0181ef73fe0',
 '4429d3bb-386c-4df0-8f21-3a060cb07122',
 '76443c74-d113-46b1-8c85-717883c3146d',
 '3fee6712-2933-4c14-8605-f519aea55a4c',
 'cb724961-6748-47df-953d-227821abe563',
 '25b6f2a3-d2a9-409d-896e-6a1e99bb03a9',
 '98ec2c4b-ced1-4b06-96c5-3f2344910a00',
 'b4fb1c5e-3db5-4e47-8cae-a7c2a98ab7f5',
 'bcaa3c95-600b-47bf-8c4d-a7fc5c31dcb5',
 'd0858bb2-b9ea-4681-bc00-72c4153284ef',
 'e1e74ad3-c393-434d-878f-1c7b01aba922',
 '015be678-11c4-4195-9ebf-63298fc820b0',
 '74d62b5e-3f42-4700-89fb-104ef5767635',
 '7ce33cdc-9bf0-4f45-b9d2-bf3c86114ba4',
 'aa584efa-066e-4866-9ffb-a8a59abda4ef',
 '9143de46-f1cc-4b9f-9c2d-69c2629ea8d1',
 'dd8ba7cc-b9ce-4202-97c8-2d6f02dfe9d5',
 '1869a19c-cecd-4a70-a1bf-8d96d1e7780c',
 'e33c4083-b279-4d6c-a74a-d84b6b897e60',
 '5bfce8ac-9849-4f69-9971-8f142ca1ecb4',
 '0beaa4de-d6c8-